[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/dlt-certified/blob/main/notebooks/day-10-capstone.ipynb)

---
# Day 10 · Capstone Project — Production dlt Pipeline
**certified-journeys / dlt-certified** &nbsp;|&nbsp; Capstone

> **Goal for today:** Build a complete, production-quality dlt pipeline from scratch. This notebook combines every concept from Days 1–9 into one working system: REST API sourcing, incremental loading, schema contracts, Pydantic validation, error handling, monitoring, and deployment.

---
## What you're building today

A production pipeline that:
- Fetches posts and users from the JSONPlaceholder REST API
- Validates records with Pydantic before loading
- Loads incrementally (only new/updated records)
- Enforces schema contracts
- Handles errors with retry logic
- Extracts pipeline metrics from `last_trace`
- Returns structured metrics for monitoring

```
JSONPlaceholder API
    │
    ▼
Pydantic validation  ←─── filter bad records with detailed logging
    │
    ▼
dlt pipeline.run()   ←─── incremental, schema_contract="evolve", merge disposition
    │
    ▼
DuckDB               ←─── raw.posts, raw.users
    │
    ▼
SQL transforms       ←─── post_metrics, user_activity views
    │
    ▼
metrics dict         ←─── rows loaded, duration, load_id, errors
```

> **Tip:** A working dlt pipeline on GitHub is worth more than any certification. Make it production-quality.

---
## Step 1 · Install dependencies

In [ ]:
%pip install -q "dlt[duckdb]" pydantic requests

---
## Step 2 · Pydantic models — define the schema contract in Python

In [ ]:
from __future__ import annotations

import logging
import os
import time
from datetime import datetime, timezone
from typing import Any, Optional

import dlt
import requests
from pydantic import BaseModel, Field, ValidationError, field_validator

# ── Logging setup ────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s [%(name)s] %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
)
logger = logging.getLogger("capstone")


# ── Pydantic models ───────────────────────────────────────────────────────────
class UserModel(BaseModel):
    """Validates a user record from JSONPlaceholder /users."""
    id:       int
    name:     str
    username: str
    email:    str
    phone:    Optional[str] = None
    website:  Optional[str] = None

    @field_validator("email")
    @classmethod
    def email_must_contain_at(cls, v: str) -> str:
        if "@" not in v:
            raise ValueError(f"Invalid email: {v!r}")
        return v.lower().strip()

    @field_validator("name", "username")
    @classmethod
    def not_empty(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("Field cannot be empty")
        return v.strip()


class PostModel(BaseModel):
    """Validates a post record from JSONPlaceholder /posts."""
    id:      int
    user_id: int    = Field(alias="userId")  # API uses camelCase
    title:   str
    body:    str

    model_config = {"populate_by_name": True}  # allow both 'userId' and 'user_id'

    @field_validator("title", "body")
    @classmethod
    def not_empty(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("Field cannot be empty")
        return v.strip()


print("Pydantic models defined: UserModel, PostModel")

# Quick smoke test
test_user = UserModel(id=1, name="Alice", username="alice99", email="Alice@Example.com")
print(f"UserModel test: {test_user.model_dump()}")

test_post = PostModel(id=1, userId=1, title="Test Post", body="Hello world")
print(f"PostModel test: {test_post.model_dump()}")

---
## Step 3 · Resource definitions with incremental loading and validation

In [ ]:
BASE_URL = "https://jsonplaceholder.typicode.com"


def fetch_with_retry(url: str, params: dict = None, max_retries: int = 3) -> list[dict]:
    """
    Fetches a JSON list from a URL with retry logic.
    Used inside resource generators.
    """
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, params=params, timeout=10)
            resp.raise_for_status()
            return resp.json()
        except requests.RequestException as e:
            wait = 2 ** attempt  # exponential backoff: 2, 4, 8 seconds
            logger.warning(f"Request failed (attempt {attempt}/{max_retries}): {e}. Retrying in {wait}s...")
            if attempt < max_retries:
                time.sleep(wait)
            else:
                raise


def validate_and_yield(records: list[dict], model, resource_name: str):
    """
    Validates records against a Pydantic model.
    Yields valid records as dicts. Logs and skips invalid ones.
    Returns (loaded, skipped) counts via a stats dict (mutated in place).
    """
    stats = {"loaded": 0, "skipped": 0}
    for raw in records:
        try:
            validated = model.model_validate(raw)
            stats["loaded"] += 1
            yield validated.model_dump()
        except ValidationError as e:
            stats["skipped"] += 1
            errors = [{"field": str(err["loc"]), "msg": err["msg"]} for err in e.errors()]
            logger.warning(f"[{resource_name}] Validation failed for id={raw.get('id')}: {errors}")
    logger.info(f"[{resource_name}] Validation complete: {stats['loaded']} loaded, {stats['skipped']} skipped")


# ── Users resource ────────────────────────────────────────────────────────────
@dlt.resource(
    primary_key="id",
    write_disposition="merge",     # upsert — safe to re-run
    schema_contract="evolve",      # allow schema evolution in dev; switch to 'freeze' in prod
)
def users():
    """
    Fetches all users from JSONPlaceholder and validates them.
    Uses merge disposition — running twice doesn't create duplicates.
    """
    logger.info("Fetching users from API...")
    raw_users = fetch_with_retry(f"{BASE_URL}/users")
    logger.info(f"Fetched {len(raw_users)} raw users")

    yield from validate_and_yield(raw_users, UserModel, "users")


# ── Posts resource with incremental loading ───────────────────────────────────
@dlt.resource(
    primary_key="id",
    write_disposition="merge",
    schema_contract="evolve",
)
def posts(
    updated_at=dlt.sources.incremental("id", initial_value=0),
):
    """
    Fetches posts incrementally. On first run: all posts.
    On subsequent runs: only posts with id > last seen id.

    Note: JSONPlaceholder doesn't support ?id_gt= params, so we filter
    client-side. In a real API, pass updated_at.last_value as a query param.
    """
    logger.info(f"Fetching posts (incremental from id={updated_at.last_value})...")
    raw_posts = fetch_with_retry(f"{BASE_URL}/posts")

    # Client-side incremental filter — in production use API params
    # e.g., fetch_with_retry(url, params={"_since": updated_at.last_value})
    new_posts = [p for p in raw_posts if p["id"] > (updated_at.last_value or 0)]
    logger.info(f"Fetched {len(raw_posts)} total, {len(new_posts)} new since last run")

    yield from validate_and_yield(new_posts, PostModel, "posts")


# ── Source: groups users + posts ──────────────────────────────────────────────
@dlt.source(name="jsonplaceholder")
def jsonplaceholder_source():
    """Complete JSONPlaceholder source: users + posts."""
    return users, posts


print("Resources defined: users(), posts(), jsonplaceholder_source()")

---
## Step 4 · The production pipeline function

In [ ]:
def build_monitoring_report(pipeline) -> dict:
    """Extracts structured metrics from pipeline.last_trace."""
    trace = pipeline.last_trace
    report = {"pipeline_name": pipeline.pipeline_name}

    if not trace:
        report["status"] = "no_trace"
        return report

    load_info = trace.last_load_info
    if load_info:
        report["load_id"]         = load_info.load_id
        report["has_failed_jobs"] = load_info.has_failed_jobs
        report["status"]          = "failed" if load_info.has_failed_jobs else "success"
        if load_info.started_at and load_info.finished_at:
            report["load_duration_sec"] = round(
                (load_info.finished_at - load_info.started_at).total_seconds(), 3
            )

    norm = trace.last_normalize_info
    if norm:
        report["row_counts"] = dict(norm.row_counts)
        report["total_rows"] = sum(norm.row_counts.values())

    step_timings = {}
    for step in trace.steps:
        if step.finished_at and step.started_at:
            step_timings[step.step] = round(
                (step.finished_at - step.started_at).total_seconds(), 3
            )
    report["step_timings_sec"] = step_timings

    return report


def apply_post_load_transforms(pipeline) -> None:
    """
    Applies SQL transforms after loading.
    Creates analytics-ready summary tables from the raw data.
    """
    TRANSFORMS = [
        (
            "post_metrics",
            """
            CREATE OR REPLACE TABLE jsonplaceholder.post_metrics AS
            SELECT
                u.id                                AS user_id,
                u.name                              AS user_name,
                u.email,
                COUNT(p.id)                         AS total_posts,
                AVG(LENGTH(p.body))                 AS avg_body_length,
                MAX(p.id)                           AS latest_post_id
            FROM jsonplaceholder.users u
            LEFT JOIN jsonplaceholder.posts p ON p.user_id = u.id
            GROUP BY u.id, u.name, u.email
            ORDER BY total_posts DESC
            """
        ),
        (
            "prolific_authors",
            """
            CREATE OR REPLACE TABLE jsonplaceholder.prolific_authors AS
            SELECT user_id, user_name, email, total_posts
            FROM jsonplaceholder.post_metrics
            WHERE total_posts >= 10
            ORDER BY total_posts DESC
            """
        ),
    ]

    with pipeline.sql_client() as client:
        for transform_name, sql in TRANSFORMS:
            try:
                client.execute_sql(sql)
                logger.info(f"Transform applied: {transform_name}")
            except Exception as e:
                logger.error(f"Transform failed [{transform_name}]: {e}")
                raise


def run_capstone_pipeline(environment: str = "development") -> dict[str, Any]:
    """
    Complete production pipeline:
    1. Extract + validate from JSONPlaceholder API
    2. Load to DuckDB with incremental merge
    3. Apply post-load SQL transforms
    4. Return structured metrics

    This function is safe to call from:
    - GitHub Actions (cron)
    - Airflow @task()
    - Prefect @task
    - Manual invocation
    """
    started_at = time.time()
    run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

    logger.info(f"Capstone pipeline starting. run_id={run_id} environment={environment}")

    metrics: dict[str, Any] = {
        "run_id":         run_id,
        "environment":    environment,
        "started_at":     datetime.now(timezone.utc).isoformat(),
        "success":        False,
        "rows_loaded":    {},
        "total_rows":     0,
        "load_id":        None,
        "duration_sec":   0.0,
        "error":          None,
        "step_timings":   {},
    }

    try:
        # ── 1. Create pipeline ──────────────────────────────────────────────
        pipeline = dlt.pipeline(
            pipeline_name="capstone",
            destination="duckdb",
            dataset_name="jsonplaceholder",
        )
        logger.info("Pipeline object created.")

        # ── 2. Run the load ─────────────────────────────────────────────────
        info = pipeline.run(jsonplaceholder_source())
        logger.info(f"Load complete. load_id={info.load_id} failed_jobs={info.has_failed_jobs}")

        # ── 3. Capture load metrics ─────────────────────────────────────────
        report = build_monitoring_report(pipeline)
        metrics["load_id"]      = report.get("load_id")
        metrics["rows_loaded"]  = report.get("row_counts", {})
        metrics["total_rows"]   = report.get("total_rows", 0)
        metrics["step_timings"] = report.get("step_timings_sec", {})

        if info.has_failed_jobs:
            raise RuntimeError(f"dlt reported failed jobs. load_id={info.load_id}")

        # ── 4. Post-load SQL transforms ──────────────────────────────────────
        apply_post_load_transforms(pipeline)
        logger.info("Post-load transforms complete.")

        metrics["success"] = True
        logger.info(
            f"Pipeline succeeded. run_id={run_id} total_rows={metrics['total_rows']} "
            f"load_id={metrics['load_id']}"
        )

    except Exception as e:
        metrics["error"]   = f"{type(e).__name__}: {e}"
        metrics["success"] = False
        logger.error(f"Pipeline FAILED. run_id={run_id} error={e}", exc_info=True)

    finally:
        metrics["duration_sec"] = round(time.time() - started_at, 3)
        logger.info(
            f"Pipeline finished. run_id={run_id} success={metrics['success']} "
            f"duration={metrics['duration_sec']}s"
        )

    return metrics


print("Pipeline function defined: run_capstone_pipeline()")

---
## Step 5 · Run the complete pipeline

In [ ]:
# Run the full pipeline — this hits the real JSONPlaceholder API
result = run_capstone_pipeline(environment="development")

print("\n" + "="*60)
print("PIPELINE RUN COMPLETE")
print("="*60)
for k, v in result.items():
    print(f"  {k:<20}: {v}")

In [ ]:
# Inspect the loaded data and transforms
pipeline = dlt.attach(pipeline_name="capstone")  # re-attach to existing pipeline state

with pipeline.sql_client() as client:
    # All tables created
    with client.execute_query(
        "SELECT table_name FROM information_schema.tables "
        "WHERE table_schema = 'jsonplaceholder' ORDER BY table_name"
    ) as cur:
        tables = [row[0] for row in cur.fetchall()]
        print("Tables in jsonplaceholder dataset:")
        for t in tables:
            print(f"  jsonplaceholder.{t}")

    # Raw data counts
    print("\n── Raw data ──")
    for tbl in ["users", "posts"]:
        with client.execute_query(f"SELECT COUNT(*) as cnt FROM jsonplaceholder.{tbl}") as cur:
            print(f"  {tbl}: {cur.df().iloc[0]['cnt']} rows")

    # Post-load transform results
    print("\n── post_metrics (top 5 authors) ──")
    with client.execute_query(
        "SELECT user_id, user_name, total_posts, ROUND(avg_body_length, 0) as avg_body "
        "FROM jsonplaceholder.post_metrics ORDER BY total_posts DESC LIMIT 5"
    ) as cur:
        print(cur.df().to_string(index=False))

    print("\n── prolific_authors (>=10 posts) ──")
    with client.execute_query(
        "SELECT user_id, user_name, email, total_posts FROM jsonplaceholder.prolific_authors"
    ) as cur:
        print(cur.df().to_string(index=False))

**What just happened?**
- All 10 users and 100 posts were fetched, validated with Pydantic, and loaded via `merge` disposition
- `incremental` tracked the highest `id` seen — the next run will only fetch new posts
- Post-load SQL transforms built `post_metrics` and `prolific_authors` analytics tables
- Structured metrics were returned — ready to send to a monitoring system

---
## Step 6 · Run it again — verify incremental loading works

In [ ]:
print("Running pipeline a second time — should load 0 new posts (all already seen)...")
result2 = run_capstone_pipeline(environment="development")

print(f"\nSecond run results:")
print(f"  success:     {result2['success']}")
print(f"  total_rows:  {result2['total_rows']}")
print(f"  rows_loaded: {result2['rows_loaded']}")
print(f"  duration:    {result2['duration_sec']}s")
print("\nNote: posts row count should be 0 or small — incremental state prevented re-loading all 100 posts")

---
## Step 7 · Production readiness checklist

Work through this checklist before publishing your pipeline to GitHub. Check off each item.

### Functionality
- [ ] Pipeline runs end-to-end without errors on first run
- [ ] Pipeline is idempotent — running twice produces the same result
- [ ] Incremental loading works — second run loads 0 rows (or only new rows)
- [ ] All tables are present and row counts are as expected
- [ ] Post-load transforms produce correct aggregations

### Data quality
- [ ] Pydantic models cover all required fields with appropriate types
- [ ] Invalid records are logged and skipped, not silently dropped
- [ ] Schema contract is set (`evolve` for dev, consider `freeze` for prod)
- [ ] Primary keys are defined on all resources
- [ ] `write_disposition` is explicit on every resource

### Error handling
- [ ] All exceptions are caught and stored in the metrics dict
- [ ] HTTP requests have retry logic with exponential backoff
- [ ] Pipeline function never raises — returns `success: False` instead
- [ ] Failure alert mechanism is in place (Slack, PagerDuty, email)

### Secrets & configuration
- [ ] No credentials hardcoded in any Python file
- [ ] `.dlt/secrets.toml` is in `.gitignore`
- [ ] Environment variable names documented in README
- [ ] GitHub Actions secrets are configured for production environment

### Observability
- [ ] Structured logging throughout (not `print()`)
- [ ] `pipeline.last_trace` metrics are extracted and returned
- [ ] Row counts, duration, and load_id are in the metrics dict
- [ ] Monitoring destination is configured (log aggregation, Datadog, etc.)

### Deployment
- [ ] `requirements.txt` or `pyproject.toml` pinned with specific versions
- [ ] GitHub Actions workflow is present (`.github/workflows/pipeline.yml`)
- [ ] Cron schedule is correct for your SLA
- [ ] `workflow_dispatch` is enabled for manual re-runs
- [ ] Artifact upload step captures logs on failure

### Documentation
- [ ] README explains what the pipeline does and how to run it locally
- [ ] Environment variables are documented
- [ ] Data model (tables + columns) is documented
- [ ] Known limitations are noted

---
## Step 8 · Portfolio README template

Copy this to your GitHub repository's `README.md` and fill in the blanks:

---

```markdown
# [Your Pipeline Name]

> A production-quality data pipeline built with [dlt](https://dlthub.com/) that loads
> [describe data source] into [destination] with incremental loading, schema contracts,
> and automated scheduling.

## What it does

- Extracts [describe data] from [source name / URL]
- Validates records with Pydantic before loading
- Loads incrementally — only new/updated records on each run
- Applies post-load SQL transforms to build analytics-ready tables
- Runs automatically every [schedule] via GitHub Actions
- Sends Slack alerts on failure

## Data model

| Table | Description | Rows (approx) |
|---|---|---|
| `raw.users` | User records from /users endpoint | ~10 |
| `raw.posts` | Post records from /posts endpoint | ~100 |
| `analytics.post_metrics` | Posts per user with engagement metrics | ~10 |

## Setup

```bash
# 1. Clone and install
git clone https://github.com/YOUR_USERNAME/YOUR_REPO
cd YOUR_REPO
pip install -r requirements.txt

# 2. Configure secrets (copy and edit)
cp .dlt/secrets.toml.example .dlt/secrets.toml
# Edit .dlt/secrets.toml with your credentials

# 3. Run locally
python pipeline.py
```

## Environment variables

| Variable | Required | Description |
|---|---|---|
| `SOURCES__MY_API__API_KEY` | Yes | API key for the source |
| `DESTINATION__BIGQUERY__PROJECT_ID` | Yes (prod) | GCP project ID |
| `SLACK_WEBHOOK_URL` | No | Slack alert on failure |
| `ENVIRONMENT` | No | `development` or `production` |

## Pipeline architecture

```
Source API → Pydantic validation → dlt pipeline → DuckDB/BigQuery
                                        ↓
                              SQL post-load transforms
                                        ↓
                              Analytics-ready tables
```

## Scheduling

Runs daily at 06:00 UTC via [GitHub Actions](.github/workflows/pipeline.yml).
Manual runs: Actions tab → Run workflow.

## Built with

- [dlt](https://dlthub.com/) — data loading
- [DuckDB](https://duckdb.org/) — local/dev destination
- [Pydantic](https://docs.pydantic.dev/) — data validation
- [GitHub Actions](https://github.com/features/actions) — scheduling

---
*Built as part of the [dlt Certified](https://certified-journeys.github.io/courses/dlt-certified/) course.*
```

---
## Challenge — complete the capstone

Your final challenge combines everything:

1. Add a `comments` resource to `jsonplaceholder_source()` using `rest_api_source` — fetch `/comments` and validate with a `CommentModel` (fields: `id`, `postId`→`post_id`, `name`, `email`, `body`)
2. Add a third post-load transform: `comment_stats` — count of comments per post, average body length
3. Update `run_capstone_pipeline()` to also report the comments row count
4. Run the full pipeline and verify 3 raw tables + 3 transform tables exist

In [ ]:
# Your solution here


<details>
<summary>Show solution</summary>

```python
from pydantic import BaseModel, Field
from typing import Optional

class CommentModel(BaseModel):
    id:      int
    post_id: int  = Field(alias="postId")
    name:    str
    email:   str
    body:    str
    model_config = {"populate_by_name": True}

@dlt.resource(primary_key="id", write_disposition="merge", schema_contract="evolve")
def comments():
    raw = fetch_with_retry(f"{BASE_URL}/comments")
    yield from validate_and_yield(raw, CommentModel, "comments")

@dlt.source(name="jsonplaceholder")
def full_source():
    return users, posts, comments

# Add to apply_post_load_transforms:
COMMENT_STATS_SQL = """
CREATE OR REPLACE TABLE jsonplaceholder.comment_stats AS
SELECT
    post_id,
    COUNT(*) AS comment_count,
    ROUND(AVG(LENGTH(body)), 0) AS avg_body_length
FROM jsonplaceholder.comments
GROUP BY post_id ORDER BY comment_count DESC
"""

# Then run:
pipeline = dlt.pipeline(
    pipeline_name="capstone_full",
    destination="duckdb",
    dataset_name="jsonplaceholder",
)
info = pipeline.run(full_source())
with pipeline.sql_client() as client:
    client.execute_sql(COMMENT_STATS_SQL)
    with client.execute_query("SELECT COUNT(*) FROM jsonplaceholder.comments") as cur:
        print("Comments:", cur.df().iloc[0][0])  # should be 500
```
</details>

---
## Day 10 key concepts recap

| Concept | What to remember |
|---|---|
| `@dlt.source` | Groups multiple resources — call `source()` to run all of them |
| `merge` + `primary_key` | Upsert pattern — idempotent, safe to re-run |
| `dlt.sources.incremental` | Persists cursor state between runs — only new records |
| `schema_contract` | `evolve` in dev, consider `freeze` or `discard_rows` in prod |
| Pydantic + generators | Validate before dlt sees data — cleanest quality layer |
| `apply_post_load_transforms()` | Idempotent SQL transforms with `CREATE OR REPLACE` |
| `build_monitoring_report()` | Structured metrics from `pipeline.last_trace` |
| Production pipeline fn | Returns metrics dict, catches all exceptions, logs everything |
| Retry with backoff | `2 ** attempt` seconds — prevents hammering failing APIs |
| `dlt.attach()` | Re-attach to existing pipeline state without re-creating it |

---
## Congratulations — you've completed the dlt Certified course!

You've built a complete production data pipeline from scratch. Here's what you can do now:

### Immediate next steps

1. **Publish your capstone to GitHub** — use the README template above
2. **Add a real data source** — replace JSONPlaceholder with an API you actually use
3. **Set up GitHub Actions** — use the workflow from Day 8 to schedule daily runs
4. **Add dbt** — use the integration from Day 6 to build proper data models

### Continue learning

| Course | What you'll learn |
|---|---|
| **dbt Certified** | SQL transformations, testing, documentation, CI/CD |
| **Terraform Certified** | Infrastructure as code for your data warehouse |
| **Airflow Certified** | Orchestration, DAG design, complex dependencies |
| **dlt advanced** | Custom destinations, state management, secrets backends |

### Skills you've demonstrated

- Building production-grade Python data pipelines with dlt
- REST API integration with authentication and pagination
- Incremental loading with stateful cursor tracking
- Data validation with Pydantic
- Schema contracts and evolution management
- Post-load SQL transformations
- Error handling, retry logic, and alerting
- Pipeline monitoring with structured metrics
- Deployment to GitHub Actions with secrets management
- dbt integration for the full ELT pattern

> **Well done.** A working pipeline on GitHub with incremental loading, validation, and monitoring is genuinely more valuable than most BI certifications. Ship it.

---
Return to your [course tracker](../index.html).